# 🧪 PT-W2-D7 概念实验：组装 MI CRE Ontology Graph v0.1

> 配套阅读：`PT-W2-D7-组装-MI-CRE-Ontology-Graph-v0.1.md`
> 把六天抽取的六维度语义构件组装成一张可遍历的语义图。

## 第 1 格：六维度数据模型

In [ ]:
from dataclasses import dataclass, field
from collections import defaultdict

@dataclass(frozen=True)
class Node:
    dimension: str  # Entity|Relationship|Lifecycle|Rule|Capability|Policy
    name: str
    attrs: dict = field(default_factory=dict)

# 六维度节点
nodes = [
    Node("Entity", "Resource Unit", {"type": "空间实体", "owner": "Asset Foundation"}),
    Node("Entity", "Contract", {"type": "契约实体", "owner": "Contract Lifecycle"}),
    Node("Entity", "Merchant", {"type": "主体实体", "owner": "Merchant"}),
    Node("Entity", "Bill/AR", {"type": "财务实体", "owner": "Billing & AR"}),
    Node("Relationship", "is_signed_by", {"category": "身份引用"}),
    Node("Relationship", "applies_to", {"category": "身份引用"}),
    Node("Relationship", "activates_occupancy", {"category": "生命周期影响"}),
    Node("Lifecycle", "Resource七态", {"states": "planned→created→available→reserved→in-use→suspended→retired"}),
    Node("Lifecycle", "Contract六态", {"states": "draft→signed→active→expiring→expired→terminated→voided"}),
    Node("Rule", "RULE-CON-024", {"type": "推理规则", "when": "合同终止时"}),
    Node("Capability", "lease.terminate", {"domain": "合同管理"}),
    Node("Capability", "lease.create", {"domain": "合同管理"}),
    Node("Policy", "合同终止审批", {"review_gate": "mandatory", "scopes": ["lease:write"]}),
]

# 六维度交叉边
edges = [
    ("Entity:Contract", "Relationship:is_signed_by", "签约关系"),
    ("Entity:Contract", "Lifecycle:Contract六态", "拥有状态机"),
    ("Lifecycle:Contract六态", "Relationship:activates_occupancy", "迁移触发关系"),
    ("Relationship:activates_occupancy", "Entity:Resource Unit", "影响目标"),
    ("Rule:RULE-CON-024", "Lifecycle:Contract六态", "守卫迁移"),
    ("Rule:RULE-CON-024", "Capability:lease.terminate", "触发动作"),
    ("Capability:lease.terminate", "Policy:合同终止审批", "受Policy约束"),
]

for e in edges:
    print(f"  {e[0]:<30} --[{e[2]}]--> {e[1]}")

## 第 2 格：BFS 遍历 Ontology Graph——从任意节点出发

In [ ]:
from collections import deque, defaultdict

graph = defaultdict(list)
for src, tgt, label in edges:
    graph[src].append((tgt, label))
    graph[tgt].append((src, label))

def traverse(start, graph, max_depth=4):
    visited = {start}
    queue = deque([(start, 0, [start])])
    results = []
    while queue:
        node, depth, path = queue.popleft()
        if depth >= max_depth:
            continue
        for neighbor, label in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                new_path = path + [f" --[{label}]--> ", neighbor]
                results.append((neighbor, depth+1, "".join(new_path)))
                queue.append((neighbor, depth+1, new_path))
    return results

print("从 Entity:Resource Unit 出发 BFS 遍历 Ontology Graph：")
for node, d, path in traverse("Entity:Resource Unit", graph):
    print(f"  depth={d} {path}")

## 第 3 格：A101 六维度交叉引用

In [ ]:
a101 = {
    "Entity": {"type": "空间实体", "state": "in-use"},
    "Identity": {"path": "龙湖天街→A栋→1F→A101", "owner": "Asset Foundation"},
    "Relationship": [
        ("is_located_in", "Floor 1F"),
        ("is_referenced_by", "Contract CT001"),
    ],
    "Lifecycle": {"machine": "Resource七态", "current": "in-use"},
    "Event": {"trigger": "ContractActivated", "effect": "occupancy-effect"},
    "Rule": ["active合同时不可出租", "退场需inspection完成"],
    "Capability": ["创建退场工单", "查询铺位状态"],
    "Policy": {"scopes": ["lease:write", "asset:read"], "review": "mandatory"},
}

for dim, val in a101.items():
    print(f"[{dim}]")
    print(f"  {val}")
    print()

## 第 4 格：可视化——六维度成熟度雷达图

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc")
font_name = font_manager.FontProperties(fname="/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc").get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

dims = ["Entity", "Relation-\nship", "Lifecycle", "Rule", "Capabil-\nity", "Policy"]
maturity = [4, 3, 4, 2, 3, 2]  # ★评分
angles = np.linspace(0, 2*np.pi, len(dims), endpoint=False).tolist()
maturity_plot = maturity + maturity[:1]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.fill(angles, maturity_plot, alpha=0.25, color="#2196F3")
ax.plot(angles, maturity_plot, "o-", color="#1565C0", lw=2)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(dims, fontsize=9)
ax.set_ylim(0, 5)
ax.set_yticks([1,2,3,4,5])
ax.set_title("MI CRE Ontology 六维度成熟度 v0.1", pad=20)
plt.tight_layout()
plt.savefig("/root/learning-notebooks/第10周/d7_ontology_graph.png", dpi=100)
plt.show()
print("成熟度雷达图已绘制")

## 第 5 格：结论——Ontology Graph 的价值

In [ ]:
print("""MI CRE Ontology Graph v0.1 组装完成。

Agent 可以：
  1. 从任意 Entity 出发，沿六维度遍历找到所有关联信息
  2. 回答 "A101 为什么不能出租？" → Lifecycle + Rule + Event
  3. 回答 "需要做什么才能出租？" → Capability + Policy
  4. 回答 "谁有权操作？" → Policy.required_scopes

关键认知变化：
  - 以前：6 套资产各管各的
  - 现在：Ontology Graph = 六维度交叉引用的语义图
  - AI Agent 读图推理，不读代码 if-else
""")